# Analyse von *YouTube*-Daten, die mit `vosonSML` gesammelt wurden

In diesem Notebook schauen wir uns exemplarisch an, wie man *YouTube*-Daten analysieren kann, die mit dem `R`-Paket [`vosonSML`](https://vosonlab.github.io/vosonSML/) gesammelt wurden.

Ausgangspunkt ist eine `.csv`-Datei, die nach der Sammlung von *YouTube*-Daten mit `vosonSML` exportiert wurde.

*Hinweis*: Als Alternative zu `vosonSML` kann für die Sammlung von *YouTube*-Daten auch das `R`-Paket [`tuber`](https://gojiplus.github.io/tuber/) genutzt werden.

## Datenimport

Zur Erinnerung: Die verwendete `.csv`-Datei muss über den Upload-Button (nach oben zeigender Pfeil) im File Explorer auf der linken Seite der Jupyter-Lab-Analyseumgebung hochgeladen werden.

In [ ]:
library(readr)

In [ ]:
yt <- read_csv("INSERT_FILE_NAME_HERE") # Namen der entsprechenden Datei (inkl. Dateiendung) einfügen

In [ ]:
names(yt)

In [ ]:
library(dplyr)

In [ ]:
glimpse(yt)

## Textanalyse

### Preprocessing

Für die Aufbereitung der Textdaten aus den *YouTube*-Kommentaren nutzen wir das Paket [`tubecleanR`](https://gesiscss.github.io/tubecleanR/). Zur Nutzung dieses Pakets gibt es in der *KODAQS Toolbox* von GESIS auch ein [ausführliches Tutorial](https://kodaqs-toolbox.gesis.org/github.com/gesiscss/tubecleanr_tool/index/).

Die Hauptfunktion des Pakers `tubecleanR` ist `parse_yt_comments()`. Diese benötigt als Input einen Datensatz (`data.frame`) mit *YouTube*-Kommentaren, die mit `vosonSML` oder `tuber` gesammelt wurden. Der Output der Funktion ist ein `data.frame`, in dem URLs/Links, Video-Zeitstempel, User Mentions, Emoticons und Emoji aus den Kommentaren in separate Spalten extrahiert wurden. Darüber hinaus erstellt die Funktion eine Spalte mit textuellen Beschreibungen der Emoji und eine weitere Spalte mit einer bereinigten Version des Kommentars, in der die zuvor aufgeführten Elemente sowie Zahlen und Satzzeichen entfernt wurden.

In [ ]:
processed_yt <- parse_yt_comments(yt,
                                  package = "vosonSML") # Paket, mit dem die Daten gesammelt wurden (vosonSML oder tuber)

In [ ]:
glimpse(processed_yt)

Die bereinigten Kommentardaten lassen sich natürlich auch wieder als `.csv`-Datei exportieren.

In [ ]:
write_csv(processed_yt, "./processed_yt.csv")

### Anzahl von Kommentare über die Zeit

Mit diesen bereinigten Textdaten lassen sich nun einige explorative bzw. deskriptive Analysen durchführen. Z.B. ist es möglich, die (kumulative) Anzahl von Kommentaren über die Zeit zu visualisieren.

In [ ]:
library(ggplot2)

In [ ]:
processed_yt |>
  filter(!is.na(Published)) |>
  mutate(date = as.Date(Published)) |>
  count(date) |>
  arrange(date) |>
  mutate(cumulative_comments = cumsum(n)) |>
  ggplot(aes(x = date, y = cumulative_comments)) +
  geom_line(color = "steelblue", size = 1) +
  labs(
    title = "Cumulative Number of YouTube Comments Over Time",
    x = "Date",
    y = "Cumulative Comments"
  ) +
  theme_minimal()

### Häufigste Emoji

Anhand der bereinigten Daten lässt sich auch analysieren, welche Emoji in den Kommentaren am häufigsten genutzt werden.

In [ ]:
processed_yt |>
  unnest_longer(Emoji) |>
  filter(!is.na(Emoji)) |>
  count(Emoji, sort = TRUE) |>
  slice_max(n, n = 15) |>
  ggplot(aes(x = reorder(Emoji, n), y = n)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +
  labs(title = "Top 15 Emoji", x = "Emoji", y = "Frequency") +
  theme_minimal()

### Erstellung eines Corpus

Wir können mit den Textdaten aus den *YouTube*-Kommentaren auch die zuvor schon für andere Plattformen genutzten Textanalysen mit dem Paket [`quanteda`](https://quanteda.io/) durchführen. Hierfür müssen wir die üblichen Aufbereitungsschritte durchlaufen: Erstellung eines Corpus, Tokenisierung + Entfernung von Stoppwörtern, Erstellung einer Document-Feature-Matrix (DFM).

In [ ]:
library(quanteda)

In [ ]:
yt_corpus <- processed_yt |>
  select(CommentID, CleanedText, 
         Published, LikeCount, Author, VideoID) |>
  corpus(docid_field = "CommentID",
         text_field = "CleanedText")

### Tokenisierung & Entfernung von Stoppwörtern

In [ ]:
tokens_yt <- tokens(yt_corpus)

In [ ]:
tokens_yt <- tokens_remove(tokens_yt,
                              stopwords("de")) |>
  tokens_remove("dass")

Für unsere Analysen wollen wir zudem nur Tokens mit einer minimalen Länge von drei Zeichen aufnehmen.

In [ ]:
tokens_yt <- tokens_select(tokens_yt,
                           min_nchar = 3)

### Document-Feature-Matrix (DFM) erstellen

In [ ]:
dfm_insta <- dfm(tokens_yt)

### Textdaten explorieren

In [ ]:
library(quanteda.textstats)

#### Worthäufigkeiten

In [ ]:
dfm_yt |>
  textstat_frequency(n = 20)

## Visualisierung

In [ ]:
library(quanteda.textplots)

#### Wortwolke

In [ ]:
dfm_yt |>
dfm_trim(min_termfreq = 10) %>%
  textplot_wordcloud()

#### Plot zu Worthäufigkeiten

In [ ]:
library(ggplot2)

In [ ]:
tstat_freq <- dfm_yt %>% 
  textstat_frequency(n = 20)

ggplot(tstat_freq, aes(x = frequency, y = reorder(feature, frequency))) +
  geom_col() + 
  labs(x = "Frequency", y = "Feature") +
  scale_x_continuous(expand = expansion(mult = c(0, 0.05)))

## Netzwerkanalyse

Die einfachste Möglichkeit für Netzwerkanalysen von *YouTube*-Daten in `R` ist die Nutzung der entsprechenden Funktionen aus dem `vosonSML`-Paket. Erklärungen dazu finden sich in der [Dokumentation des Pakets](https://vosonlab.github.io/vosonSML/) sowie der Vignette ["Introduction to vosonSML"](https://vosonlab.github.io/vosonSML/articles/Intro-to-vosonSML.html).

Die Funktionen aus `vosonSML`sind kompatibel mit dem Netzwerkanalysepaket [`igraph`](https://r.igraph.org/).

**Wichtig**: Die `vosonSML`-Funktionen für die Netzwerkanalyse funktionieren nur mit einem Datenobjekt, das mit dem Paket erstellt wurde. D.h. eine importierte `.csv`-Datei ist in diesem Fall keine Option.